In [21]:
import duckdb
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [22]:
boundary_file = "../data/raw/boundary/uae_boundary.geojson"

uae_boundary = gpd.read_file(boundary_file).to_crs("EPSG:4326")

ookla_files = [
    "../data/raw/ookla/2024-07-01_performance_mobile_tiles.parquet",
    "../data/raw/ookla/2024-10-01_performance_mobile_tiles.parquet",
    "../data/raw/ookla/2025-01-01_performance_mobile_tiles.parquet",
    "../data/raw/ookla/2025-04-01_performance_mobile_tiles.parquet",
    "../data/raw/ookla/2025-07-01_performance_mobile_tiles.parquet",
    "../data/raw/ookla/2025-10-01_performance_mobile_tiles.parquet",
    "../data/raw/ookla/2026-01-01_performance_mobile_tiles.parquet",
    "../data/raw/ookla/2026-04-01_performance_mobile_tiles.parquet",
]

In [23]:
def extract_uae_tiles(file):

    # Step 1: rough geographic pre-filter
    candidate = duckdb.sql(f"""
        SELECT *
        FROM read_parquet('{file}')
        WHERE tile_x BETWEEN 51 AND 57
          AND tile_y BETWEEN 22 AND 27
    """).df()

    # Step 2: convert centroids to geographic points
    points = gpd.GeoDataFrame(
        candidate,
        geometry=gpd.points_from_xy(
            candidate["tile_x"],
            candidate["tile_y"]
        ),
        crs="EPSG:4326"
    )

    # Step 3: exact UAE polygon filter
    uae = gpd.sjoin(
        points,
        uae_boundary[["geometry"]],
        predicate="within",
        how="inner"
    )

    # remove spatial-join helper column
    if "index_right" in uae.columns:
        uae = uae.drop(columns=["index_right"])

    return uae

In [25]:
# Process all 8

all_quarters = []

for file in ookla_files:

    print("Processing:", Path(file).name)

    uae = extract_uae_tiles(file)

    print("UAE tiles:", len(uae))

    all_quarters.append(uae)

Processing: 2024-07-01_performance_mobile_tiles.parquet
UAE tiles: 5818
Processing: 2024-10-01_performance_mobile_tiles.parquet
UAE tiles: 6023
Processing: 2025-01-01_performance_mobile_tiles.parquet
UAE tiles: 5965
Processing: 2025-04-01_performance_mobile_tiles.parquet
UAE tiles: 5982
Processing: 2025-07-01_performance_mobile_tiles.parquet
UAE tiles: 6187
Processing: 2025-10-01_performance_mobile_tiles.parquet
UAE tiles: 6844
Processing: 2026-01-01_performance_mobile_tiles.parquet
UAE tiles: 7162
Processing: 2026-04-01_performance_mobile_tiles.parquet
UAE tiles: 6879


In [26]:
# Combine all 8 
ookla_uae = pd.concat(
    all_quarters,
    ignore_index=True
)

In [ ]:
ookla_uae.shape

In [29]:
Path("../data/processed").mkdir(exist_ok=True)

ookla_uae.to_parquet(
    "../data/processed/ookla_tiles_uae.parquet",
    index=False
)